# Hàm dùng chung

Các notebook theo giai đoạn tự nạp những hàm cần thiết ở đây. Chỉnh hàm tại mục tương ứng. Các ô mốc đối chiếu lưu kết quả lịch sử phục vụ kiểm tra lại từ CSV.

In [ ]:
"""Đọc các ô mã dùng chung trong notebook và nạp theo tên khi được import."""

import importlib.abc
import importlib.util
import io
import json
import os
from pathlib import Path
import sys


class NotebookResource:
    """Mốc đối chiếu lịch sử được lưu trong notebook, truy cập chỉ đọc."""

    def __init__(self, name):
        self.name = name

    def read_bytes(self):
        return resources[self.name].encode("utf-8")

    def read_text(self, encoding="utf-8"):
        return self.read_bytes().decode(encoding)

    def open(self, mode="rb"):
        if mode != "rb":
            raise ValueError("Mốc đối chiếu chỉ hỗ trợ đọc byte")
        return io.BytesIO(self.read_bytes())

    def with_name(self, name):
        return NotebookResource(str(Path(self.name).with_name(name).as_posix()))

    def __str__(self):
        return self.name


class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        if fullname in sources or fullname in packages:
            return importlib.util.spec_from_loader(fullname, self, is_package=fullname in packages)
        return None

    def create_module(self, spec):
        return None

    def exec_module(self, module):
        name = module.__name__
        module.__file__ = str(SUPPORT_FILE)
        module.__dict__.update(PROJECT_ROOT=SUPPORT_FILE.parents[1], notebook_resource=NotebookResource)
        if name in sources:
            exec(compile(sources[name], f"{SUPPORT_FILE}:{name}", "exec"), module.__dict__)


if "SUPPORT_FILE" not in globals():
    starts = Path(os.environ.get("RETAIL_ANALYTICS_ROOT", Path.cwd())).resolve()
    SUPPORT_FILE = next(
        path / "notebook/00_ham_dung_chung.ipynb"
        for path in (starts, *starts.parents)
        if (path / "notebook/00_ham_dung_chung.ipynb").is_file()
    )

document = json.loads(SUPPORT_FILE.read_text(encoding="utf-8"))
sources = {}
resources = {}
packages = {"datathon", "datathon.eda", "datathon.preprocessing", "datathon.segmentation"}
for cell in document["cells"]:
    source = "".join(cell["source"])
    if "module" in cell["metadata"]:
        sources[cell["metadata"]["module"]] = source
    if "resource" in cell["metadata"]:
        resources[cell["metadata"]["resource"]] = source

if not any(getattr(finder, "notebook_file", None) == SUPPORT_FILE for finder in sys.meta_path):
    finder = NotebookModules()
    finder.notebook_file = SUPPORT_FILE
    sys.meta_path.insert(0, finder)


## Nạp dữ liệu và đối soát

## Tiền xử lý

## Phân cụm khách hàng

## Mốc đối chiếu của lần nghiệm thu trước